In [ ]:
import tensorflow as tf
import tensorflow_datasets as tfds
import tensorflow_hub as hub
import matplotlib.pyplot as plt

In [ ]:
import timm

print("Available Vision Transformer Models: ")
print(timm.list_models("vit*"))


Available Vision Transformer Models: 
['vit_base_mci_224', 'vit_base_patch8_224', 'vit_base_patch14_dinov2', 'vit_base_patch14_reg4_dinov2', 'vit_base_patch16_18x2_224', 'vit_base_patch16_224', 'vit_base_patch16_224_miil', 'vit_base_patch16_384', 'vit_base_patch16_clip_224', 'vit_base_patch16_clip_384', 'vit_base_patch16_clip_quickgelu_224', 'vit_base_patch16_gap_224', 'vit_base_patch16_plus_240', 'vit_base_patch16_plus_clip_240', 'vit_base_patch16_reg4_gap_256', 'vit_base_patch16_rope_reg1_gap_256', 'vit_base_patch16_rpn_224', 'vit_base_patch16_siglip_224', 'vit_base_patch16_siglip_256', 'vit_base_patch16_siglip_384', 'vit_base_patch16_siglip_512', 'vit_base_patch16_siglip_gap_224', 'vit_base_patch16_siglip_gap_256', 'vit_base_patch16_siglip_gap_384', 'vit_base_patch16_siglip_gap_512', 'vit_base_patch16_xp_224', 'vit_base_patch32_224', 'vit_base_patch32_384', 'vit_base_patch32_clip_224', 'vit_base_patch32_clip_256', 'vit_base_patch32_clip_384', 'vit_base_patch32_clip_448', 'vit_base_p

In [25]:
# Load CIFAR-10 from TensorFlow Datasets
dataset_name = "cifar10"
BATCH_SIZE = 128
IMG_SIZE = 224  # You can still use 224 for Vision Transformer input size

# Load dataset
train_ds, val_ds = tfds.load(dataset_name, split=['train', 'test'], as_supervised=True)

def preprocess(image, label):
    image = tf.image.resize(image, (IMG_SIZE, IMG_SIZE))  # Resize to 224x224
    image = tf.cast(image, tf.float32) / 255.0  # Normalize to [0, 1]
    return image, label

# Apply preprocessing and batching
train_ds = train_ds.map(preprocess).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
val_ds = val_ds.map(preprocess).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)


In [29]:
import timm
import tensorflow as tf
from tensorflow.keras.layers import Layer, Dense, GlobalAveragePooling2D
from tensorflow.keras.models import Model
import numpy as np

class ViT_Model(tf.keras.Model):
    def __init__(self, num_classes=1000, vit_model_name='vit_base_patch16_224'):
        super(ViT_Model, self).__init__()

        # Load the pretrained ViT model using timm
        self.vit = timm.create_model(vit_model_name, pretrained=True, num_classes=0)

        # The number of features in the last layer of ViT (embedding dimension)
        self.embed_dim = self.vit.num_features

        # We use a Global Average Pooling layer directly on ViT features
        self.pooling = GlobalAveragePooling2D()

        # Classification layer
        self.classifier = Dense(num_classes, activation='softmax')

    def call(self, inputs, training=False):
        # Resize the input to match ViT expected input size (224x224)
        inputs = tf.image.resize(inputs, (IMG_SIZE, IMG_SIZE))

        # Pass the input through ViT model to extract features
        vit_output = self.vit.forward_features(inputs)

        # ViT returns output of shape (batch_size, num_patches, embed_dim)
        # Take the first token [CLS] representation (index 0)
        cls_token = vit_output[:, 0, :]  # CLS token's embedding

        # Optionally, apply any additional processing (like pooling)
        pooled_features = self.pooling(cls_token)  # Global Average Pooling

        # Classify the pooled features
        output = self.classifier(pooled_features)

        return output

In [30]:
# Define the model
num_classes = 10  # For CIFAR-10, for instance
vit_model_name = 'vit_base_patch16_224'  # ViT model name from timm
model = ViT_Model(num_classes=num_classes, vit_model_name=vit_model_name)

# Print model summary
model.build((None, IMG_SIZE, IMG_SIZE, 3))  # Input shape: (None, 224, 224, 3)
model.summary()

# Compile the model
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=3e-5),
              loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
              metrics=['accuracy'])

# Load your dataset and preprocess images
# Assuming `train_ds` and `val_ds` are your training and validation datasets

# Train the model
EPOCHS = 10
history = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS)

# Evaluate model on validation set
val_loss, val_acc = model.evaluate(val_ds)
print(f"Validation Accuracy: {val_acc * 100:.2f}%")

/usr/local/lib/python3.11/dist-packages/keras/src/layers/layer.py:393: UserWarning: `build()` was called on layer 'vi_t__model_2', however the layer does not have a `build()` method implemented and it looks like it has unbuilt state. This will cause the layer to be marked as built, despite not being actually built, which may cause failures down the line. Make sure to implement a proper `build()` method.
  warnings.warn(


Model: "vi_t__model_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ global_average_pooling2d_2           │ ?                           │               0 │
│ (GlobalAveragePooling2D)             │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_3 (Dense)                      │ ?                           │     0 (unbuilt) │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10


AssertionError: Exception encountered when calling ViT_Model.call().

[1mInput width (3) doesn't match model (224).[0m

Arguments received by ViT_Model.call():
  • inputs=tf.Tensor(shape=(None, 224, 224, 3), dtype=float32)
  • training=True